<a href="https://colab.research.google.com/github/malik8-beep/Hello/blob/main/rnn_modeling_Malik_KONFE_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from torch.utils.data import Dataset , DataLoader
import torch
import torch.nn as nn
import torch.optim as optim
import re
import string
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv("/content/drive/MyDrive/NLP/dataset(1).csv",encoding='latin-1')
df.head()

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Category  5572 non-null   object
 1   Message   5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB


In [4]:
df.describe()

,Category,Message
count,5572,5572
unique,2,5157
top,ham,"Sorry, I'll call later"
freq,4825,30


In [5]:
# Nettoyage colonnes (si besoin)
df = df[['Category', 'Message']]  # garder seulement les 2 colonnes utile

In [6]:
# Distribution des classes
print("Nombre total d'exemples :", len(df))
print("\nRépartition des classes :")
print(df['Category'].value_counts())
print("\nPourcentages :")
print(df['Category'].value_counts(normalize=True) * 100)

Nombre total d'exemples : 5572

Répartition des classes :
Category
ham     4825
spam     747
Name: count, dtype: int64

Pourcentages :
Category
ham     86.593683
spam    13.406317
Name: proportion, dtype: float64


In [7]:
# 2. Nettoyage des labels (ham → 0, spam → 1)
df['label'] = df['Category'].map({'ham': 0, 'spam': 1})
df = df.drop('Category', axis=1)

print("Répartition après mapping :")
print(df['label'].value_counts())

Répartition après mapping :
label
0    4825
1     747
Name: count, dtype: int64


In [8]:
def clean_text(text):
    # Minuscules
    text = text.lower()

    # Supprimer les URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)

    # Supprimer les mentions @ et # (pas très utiles ici)
    text = re.sub(r'@\w+|#\w+', '', text)

    # Supprimer les caractères spéciaux, ponctuation (sauf quelques exceptions)
    text = re.sub(r'[^a-zA-Z0-9\s!?\'$£€]', '', text)

    # Remplacer plusieurs espaces par un seul
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Application
df['clean_message'] = df['Message'].apply(clean_text)

# Aperçu
print("\nExemples après nettoyage :")
print(df[['Message', 'clean_message']].head(8))


Exemples après nettoyage :
                                             Message  \
0  Go until jurong point, crazy.. Available only ...   
1                      Ok lar... Joking wif u oni...   
2  Free entry in 2 a wkly comp to win FA Cup fina...   
3  U dun say so early hor... U c already then say...   
4  Nah I don't think he goes to usf, he lives aro...   
5  FreeMsg Hey there darling it's been 3 week's n...   
6  Even my brother is not like to speak with me. ...   
7  As per your request 'Melle Melle (Oru Minnamin...   

                                       clean_message  
0  go until jurong point crazy available only in ...  
1                            ok lar joking wif u oni  
2  free entry in 2 a wkly comp to win fa cup fina...  
3        u dun say so early hor u c already then say  
4  nah i don't think he goes to usf he lives arou...  
5  freemsg hey there darling it's been 3 week's n...  
6  even my brother is not like to speak with me t...  
7  as per your request 'mel

In [9]:
!pip install tensorflow
from tensorflow.keras.preprocessing.text import Tokenizer

# Hyperparamètres souvent utilisés pour les SMS
MAX_WORDS = 8000       # taille max du vocabulaire
MAX_LEN = 150          # longueur maximale des séquences (la plupart des SMS sont courts)

# Création du tokenizer
tokenizer = Tokenizer(
    num_words=MAX_WORDS,
    filters='!"#$%&()*+,-./:;<=>?@[\]^_`{|}~\t\n',
    lower=True,
    oov_token='<OOV>'     # token pour les mots inconnus
)

# On entraîne le tokenizer sur les messages nettoyés
tokenizer.fit_on_texts(df['clean_message'])

# Conversion en séquences d'entiers
sequences = tokenizer.texts_to_sequences(df['clean_message'])

# Statistiques utiles
word_index = tokenizer.word_index
print(f"Nombre de mots uniques trouvés : {len(word_index)}")
print(f"Quelques premiers mots du vocabulaire :")
print(list(word_index.items())[:15])

<>:11: SyntaxWarning: invalid escape sequence '\]'
<>:11: SyntaxWarning: invalid escape sequence '\]'
/tmp/ipython-input-355030764.py:11: SyntaxWarning: invalid escape sequence '\]'
  filters='!"#$%&()*+,-./:;<=>?@[\]^_`{|}~\t\n',


Nombre de mots uniques trouvés : 9565
Quelques premiers mots du vocabulaire :
[('<OOV>', 1), ('to', 2), ('i', 3), ('you', 4), ('a', 5), ('the', 6), ('u', 7), ('and', 8), ('is', 9), ('in', 10), ('me', 11), ('my', 12), ('for', 13), ('your', 14), ('it', 15)]


In [10]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Padding : on ajoute des zéros pour avoir toutes les séquences de même longueur
X = pad_sequences(
    sequences,
    maxlen=MAX_LEN,
    padding='post',       # ou 'pre' - post est souvent mieux pour LSTM/GRU
    truncating='post'     # on coupe les messages trop longs à la fin
)

print("\nForme des données après padding :", X.shape)

# Exemple visuel d'une séquence paddée
print("\nExemple de séquence paddée (première ligne) :")
print(X[0])
print("Longueur après padding :", len(X[0]))


Forme des données après padding : (5572, 150)

Exemple de séquence paddée (première ligne) :
[  45  442 4389  805  726  686   63   10 1260   91  120  358 1261  153
 2932 1262   67   56 4390  138    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0]
Longueur après padding : 150


In [11]:
from sklearn.model_selection import train_test_split

# Supposons que vous avez déjà :
# X = séquences paddées (numpy array)
y = df['label'].values # Définir y à partir de la colonne 'label' de votre DataFrame

# Étape 1 : Séparation train + (val+test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,           # 30% pour val + test
    random_state=42,
    stratify=y                # très important pour garder la proportion spam/ham
)

# Étape 2 : Séparation du reste en validation et test (50/50 du temp)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,           # 50% de 30% → 15% du total
    random_state=42,
    stratify=y_temp
)

# Vérification des proportions
print("Taille totale du dataset :", len(X))
print("-" * 60)
print(f"Train:      {len(X_train):5d} échantillons ({len(X_train)/len(X)*100:5.1f}%)")
print(f"Validation: {len(X_val):5d} échantillons ({len(X_val)/len(X)*100:5.1f}%)")
print(f"Test:       {len(X_test):5d} échantillons ({len(X_test)/len(X)*100:5.1f}%)")
print("-" * 60)

# Pourcentage de spam dans chaque ensemble (doit rester proche)
print("Proportion de spam :")
print(f"  Train : {np.mean(y_train)*100:5.1f}%")
print(f"  Val   : {np.mean(y_val)*100:5.1f}%")
print(f"  Test  : {np.mean(y_test)*100:5.1f}%")

Taille totale du dataset : 5572
------------------------------------------------------------
Train:       3900 échantillons ( 70.0%)
Validation:   836 échantillons ( 15.0%)
Test:         836 échantillons ( 15.0%)
------------------------------------------------------------
Proportion de spam :
  Train :  13.4%
  Val   :  13.4%
  Test  :  13.4%


In [12]:
# Méthode en une étape (même résultat)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.15,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train,
    test_size=0.1765,          # ≈ 15/85
    random_state=42,
    stratify=y_train
)

In [13]:
class VocabularyBuilder:
  def __init__(self, dataframe):
    self.df = dataframe

    self.word_to_idx , self.idx_to_word = self.build_vocab()


  def build_vocab(self):
    vocab = set()

    for text in self.df["Message"]:
      vocab.update(text.split())

    word_to_idx = {'<pad>':0 , '<unk>':1}

    for idx, word in enumerate(sorted(vocab) , start=2):
        word_to_idx[word]=idx

    idx_to_word = {idx:word for word ,idx in word_to_idx.items()}


    return word_to_idx , idx_to_word

  def encode(self , text):
    return[self.word_to_idx.get(word, self.word_to_idx['<unk>']) for word in text.split()]


  def decode(self, indices):
    return [self.idx_to_word.get(idx) for idx in indices]

if __name__ == "__main__":
  # Using the existing 'df' DataFrame from the global scope
  Vocabuilder = VocabularyBuilder(dataframe=df)

  text = "Hello i am ok"

  indices = Vocabuilder.encode(text)


  texte = Vocabuilder.decode(indices)
  print(indices)
  print(" ".join(texte))

class SPAMDataset(Dataset):
  def __init__(self , dataframe , vocab_builder, max_length=128):
    self.df = dataframe # Assuming dataframe is already a DataFrame
    self.max_length = max_length
    self.vocab_builder = vocab_builder # Store the vocab builder instance
    self.classes_map = {'ham':0 , 'spam':1}

  def __len__(self):
    return len(self.df)

  def __getitem__(self , idx):
    text = self.df.iloc[idx]["Message"]
    label = self.df.iloc[idx]["Category"]

    input_ids = self.vocab_builder.encode(text) # Use the stored vocab_builder

    if len(input_ids) < self.max_length:
      input_ids += [0] * (self.max_length - len(input_ids))

    else:
        input_ids = input_ids[:self.max_length] # Correct truncation logic

    # Convert to tensors before returning
    return torch.tensor(input_ids, dtype=torch.long), torch.tensor(self.classes_map[label], dtype=torch.long)


[2950, 9712, 6001, 11612]
Hello i am ok


In [14]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

# Define the missing variables
vocab_size = len(word_index) + 1  # +1 for padding token
max_len = MAX_LEN                 # Already defined globally
num_classes = 2                   # Binary classification (ham/spam)

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=128),
    LSTM(128),
    Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [15]:
loss = "sparse_categorical_crossentropy"


In [16]:
from tensorflow.keras.optimizers import Adam

optimizer = Adam(learning_rate=0.001)


In [17]:
model.compile(
    optimizer=optimizer,
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


In [18]:
history = model.fit(
    X_train,
    y_train,
    batch_size=32,
    epochs=20,
    validation_split=0.2,
    verbose=1
)


Epoch 1/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 40s 366ms/step - accuracy: 0.8288 - loss: 0.4214 - val_accuracy: 0.8577 - val_loss: 0.4093
Epoch 2/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 29s 300ms/step - accuracy: 0.8749 - loss: 0.3792 - val_accuracy: 0.8577 - val_loss: 0.4092
Epoch 3/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 29s 301ms/step - accuracy: 0.8546 - loss: 0.4228 - val_accuracy: 0.8577 - val_loss: 0.4198
Epoch 4/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 30s 307ms/step - accuracy: 0.8743 - loss: 0.3787 - val_accuracy: 0.8577 - val_loss: 0.4093
Epoch 5/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 29s 298ms/step - accuracy: 0.8594 - loss: 0.4073 - val_accuracy: 0.8577 - val_loss: 0.4236
Epoch 6/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 30s 309ms/step - accuracy: 0.8729 - loss: 0.3835 - val_accuracy: 0.8577 - val_loss: 0.4091
Epoch 7/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 29s 294ms/step - accuracy: 0.8704 - loss: 0.3885 - val_accuracy: 0.8577 - val_loss: 0.4103
Epoch 8/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 29s 295ms/step - accuracy: 0.8744 - loss: 0.3819 - val_accu

In [19]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)


In [20]:
!pip install wandb
!wandb login

wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Find your API key here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter: 
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


In [ ]:
!wandb login --relogin

wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Find your API key here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter: 

In [ ]:
# 1. Compile
model.compile(
    optimizer=Adam(0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# 2. Train + Wandb
model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=32,
    callbacks=[EarlyStopping(patience=3), WandbCallback()]
)


In [ ]:
import numpy as np

y_pred_proba = model.predict(X_test)


In [ ]:
y_pred = np.argmax(y_pred_proba, axis=1)


In [ ]:
y_pred = (y_pred_proba > 0.5).astype(int)


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)


In [ ]:
accuracy = accuracy_score(y_test, y_pred)


In [ ]:
precision = precision_score(y_test, y_pred, average="macro")


In [ ]:
recall = recall_score(y_test, y_pred, average="macro")


In [ ]:
f1 = f1_score(y_test, y_pred, average="macro")


In [ ]:
print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-score  : {f1:.4f}")


In [ ]:
print(classification_report(y_test, y_pred))


In [ ]:
!wandb.log({
  "test_accuracy": accuracy,
  "test_precision": precision,
  "test_recall": recall,
  "test_f1": f1
  })
